# Phase 3: Python Data Analysis & Visualization Catalog
**Course:** ADY201m - Applied Data Analysis  
**Project:** UAE Used-Car Market Analysis & Intelligent Valuation Engine  
**Objective:** Execute SQL queries, deep Python data mining, econometric regressions, and generate all 12 publication-grade figures (Figures 3.1 to 3.12).

In [ ]:
import sys
import sqlite3
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import plotly.express as px

# Adjust project root
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DB_PATH, MODEL_FEATURES_PATH, FIGURES_DIR, 
    AED_TO_VND, setup_chart_style
)

setup_chart_style()
print("Environment initialized successfully.")

In [ ]:
# Load model feature set
df = pd.read_csv(MODEL_FEATURES_PATH)
print(f"Loaded dataset: {df.shape[0]:,} listings, {df.shape[1]} features.")
df.head(3)

In [ ]:
# Section III.1: SQLite Database Verification
conn = sqlite3.connect(DB_PATH)
q1_sql = """
SELECT 
    COUNT(*) AS total_listings,
    ROUND(AVG(price_aed), 2) AS avg_price_aed,
    MIN(price_aed) AS min_price_aed,
    MAX(price_aed) AS max_price_aed
FROM cars;
"""
q1_df = pd.read_sql_query(q1_sql, conn)
print("=== Query Q1 Output ===")
display(q1_df)

In [ ]:
# Section III.2: Table 3.3 - Cross-tabulation Body Type x Fuel Type
ct_body_fuel = pd.crosstab(df["body_type"], df["fuel_type"], margins=True)
print("=== Table 3.3. Cross-tabulation: Body Type by Fuel Type ===")
display(ct_body_fuel)

# Table 3.4: Multi-index Groupby Make Tier x Age Bracket
bins = [0, 3, 6, 10, 15, 21]
labels = ["0–3 yr", "4–6 yr", "7–10 yr", "11–15 yr", "16–20 yr"]
df["age_bracket"] = pd.cut(df["car_age"], bins=bins, labels=labels, right=True)
tier_age_grp = df.groupby(["make_tier", "age_bracket"]).agg(
    listing_count=("price_aed", "count"),
    median_price_aed=("price_aed", "median"),
    median_mileage_km=("mileage_km", "median")
).round(0)
print("
=== Table 3.4. Multi-index Groupby: Make Tier by Age Bracket ===")
display(tier_age_grp)

In [ ]:
# Table 3.5: Correlation Matrix
num_cols = ["log_price", "car_age", "log_mileage", "cylinders", "n_features"]
spearman_corr = df[num_cols].corr(method="spearman").round(3)
pearson_corr = df[num_cols].corr(method="pearson").round(3)
print("=== Table 3.5a: Spearman Rank Correlation Matrix ===")
display(spearman_corr)
print("
=== Table 3.5b: Pearson Linear Correlation Matrix ===")
display(pearson_corr)

# Within-model normalization
model_med = df.groupby(["make", "model"])["price_aed"].transform("median")
df["price_ratio"] = df["price_aed"] / model_med.replace(0, np.nan)
df_valid = df[df["price_ratio"].between(0.1, 5)].dropna(subset=["price_ratio", "car_age", "log_mileage"])

# Table 3.6: Within-Model Regression
s_age, _, r_age, p_age, _ = stats.linregress(df_valid["car_age"], df_valid["price_ratio"])
s_mil, _, r_mil, p_mil, _ = stats.linregress(df_valid["log_mileage"], df_valid["price_ratio"])
print(f"Within-Model Regression -- car_age: slope={s_age:.5f}, r2={r_age**2:.4f}, p={p_age:.4f}")
print(f"Within-Model Regression -- log_mileage: slope={s_mil:.5f}, r2={r_mil**2:.4f}, p={p_mil:.4f}")

# Table 3.7: Kruskal-Wallis Test on Vehicle Condition
groups = [grp["price_ratio"].values for _, grp in df_valid.groupby("condition") if len(grp) > 5]
kw_stat, kw_p = stats.kruskal(*groups)
print(f"
Kruskal-Wallis Condition Test: H={kw_stat:.3f}, p-value={kw_p:.4f} (Not statistically significant)")

In [ ]:
# Section III.3: Generate All 12 Publication-Grade Figures
from src.run_phase3 import run_phase3
paths, results = run_phase3()
print(f"Successfully verified and regenerated all {len(paths)} catalog figures in figures/")